In [194]:
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np 
import seaborn as sns 

In [195]:
df=pd.read_csv('car data.csv')
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [196]:
df.shape

(301, 9)

In [197]:
df.isnull().sum()

Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Kms_Driven       0
Fuel_Type        0
Seller_Type      0
Transmission     0
Owner            0
dtype: int64

In [198]:
df.duplicated().sum()

np.int64(2)

In [199]:
df.describe()

,Year,Selling_Price,Present_Price,Kms_Driven,Owner
count,301.000000,301.000000,301.000000,301.000000,301.000000
mean,2013.627907,4.661296,7.628472,36947.205980,0.043189
std,2.891554,5.082812,8.644115,38886.883882,0.247915
min,2003.000000,0.100000,0.320000,500.000000,0.000000
25%,2012.000000,0.900000,1.200000,15000.000000,0.000000
50%,2014.000000,3.600000,6.400000,32000.000000,0.000000
75%,2016.000000,6.000000,9.900000,48767.000000,0.000000
max,2018.000000,35.000000,92.600000,500000.000000,3.000000


In [200]:
numeric_cols = ['Year', 'Present_Price', 'Kms_Driven', 'Owner', 'Selling_Price']
for col in numeric_cols:
  skew=df[col].skew()
  print(f"{col}: {skew:.2f}")


Year: -1.25
Present_Price: 4.08
Kms_Driven: 6.44
Owner: 7.62
Selling_Price: 2.49


In [201]:
'''skew_features=['Present_Price', 'Kms_Driven', 'Selling_Price']
for col in skew_features:
  df[col]=np.log1p(df[col])'''

"skew_features=['Present_Price', 'Kms_Driven', 'Selling_Price']\nfor col in skew_features:\n  df[col]=np.log1p(df[col])"

In [202]:
'''for col in skew_features:
  skew=df[col].skew()
  print(f"{col}: {skew:.2f}")'''

'for col in skew_features:\n  skew=df[col].skew()\n  print(f"{col}: {skew:.2f}")'

In [203]:
df.drop(['Car_Name'], axis=1, inplace=True)
df.head()

,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [204]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
for col in ['Fuel_Type', 'Seller_Type', 'Transmission']:
  df[col]=le.fit_transform(df[col])

In [205]:
df.head()

,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,2014,3.35,5.59,27000,2,0,1,0
1,2013,4.75,9.54,43000,1,0,1,0
2,2017,7.25,9.85,6900,2,0,1,0
3,2011,2.85,4.15,5200,2,0,1,0
4,2014,4.60,6.87,42450,1,0,1,0


In [206]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold
X=df.drop('Selling_Price', axis=1)
y=df['Selling_Price']
model=GradientBoostingRegressor(random_state=42)

In [207]:
param_grid={
  'learning_rate': [0.01, 0.05, 0.1],
  'n_estimators': [50, 100, 200],
  'max_depth': [2,3,4]
}
cv=KFold(n_splits=5, shuffle=True, random_state=42)
grid_search=GridSearchCV(estimator=model,
                         param_grid=param_grid,
                         cv=cv,
                         scoring='r2',
                         n_jobs=-1,
                         verbose=2)
grid_search.fit(X,y)
print(grid_search.best_params_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}


In [208]:
from sklearn.metrics import mean_squared_error, r2_score
best_model=grid_search.best_estimator_
y_pred=best_model.predict(X)
'''y_pred_original=np.expm1(y_pred)
y_original=np.expm1(y)'''
rmse=np.sqrt(mean_squared_error(y, y_pred))
r2=r2_score(y, y_pred)
print("RMSE: ", rmse)
print("R2: ", r2)

RMSE:  0.19776053064006918
R2:  0.9984811444900188


In [209]:
import pickle as pkl
with open('car_price.pkl', 'wb') as f:
  pkl.dump((best_model, le), f)